In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import xarray as xr
import cftime
import pandas as pd
import os

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.patches import Rectangle, Ellipse
import matplotlib.colors as mcolors
import cmaps
import inspect
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

plt.rcParams['font.family'] = 'serif'         
plt.rcParams['font.serif'] = ['Times New Roman']
# plt.rcParams['text.usetex'] = True
# plt.rcParams['text.latex.preamble'] = r'\usepackage{xcolor}'

In [ ]:
alh = 'abcdefghijklmno'

In [ ]:
def plot_sst(axs, data, levels, draw_title=False, draw_text=False, cmap=cmaps.hotcold_18lev):
    for i, ax in enumerate(axs.flatten()):
        ax.set_aspect('auto')
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1.2) 
            spine.set_edgecolor('black')
        ax.set_extent([-60, 60, -40, 40], crs=ccrs.PlateCarree(central_longitude=180))
        ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#F5F5F5', lw=0.1, zorder=1)
        ax.coastlines(lw=0.6, color='k')
        major_lon = np.array([120, 150, 180, 210, 240, 270])
        major_lat = [-40, -20, 0, 20, 40]
        minor_lon = range(120, 281, 15) 
        minor_lat = range(-40, 41, 10)
        ax.set_xticks(major_lon-180, crs=ccrs.PlateCarree(central_longitude=180)) 
        ax.set_xticks(minor_lon, crs=ccrs.PlateCarree(), minor=True)
        ax.set_yticks(major_lat, crs=ccrs.PlateCarree())
        ax.set_yticks(minor_lat, crs=ccrs.PlateCarree(), minor=True)
        ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.tick_params(axis='both', which='major', direction='out',
                    length=5, width=1.2, labelsize=9, colors='black')
        ax.tick_params(axis='both', which='minor', direction='out',
                    length=2, width=1, colors='black')
        ax.tick_params(axis='both', labelsize=9)
        if draw_title:
            ax.set_title(titles[i], fontsize=12)
        if draw_text:
            ax.text(-0.02, 1.05, texts[i], transform=ax.transAxes, fontsize=16, fontweight='bold')
        cb = ax.contourf(lon, lat, data[i], levels=levels, cmap=cmap, transform=ccrs.PlateCarree(), extend='both')
        # LAT, LON = np.meshgrid(lat, lon, indexing='ij')
        # mask = pvalue[i] <= 0.05
        # skip = 4
        # ax.scatter(LON[mask][::skip, ::skip], LAT[mask][::skip, ::skip], color='k', s=2, transform=ccrs.PlateCarree())
    return cb

In [ ]:
nrows, ncols = 4, 3
wsize, hsize = 9, 6
fig, axs = plt.subplots(nrows, ncols, figsize=(wsize, hsize), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})

ds_nino = xr.open_dataset('./data/composite_dTdt.nc')
# ds_nino = xr.open_dataset('./data-d4pdf/composite_tc_sst.nc')
# ds_nino_obs = xr.open_dataset('./data-20c/composite_tc_sst_obs.nc')

sst_start, sst_end = 16,21
Ntc_start, Ntc_end = 5, 10
Stc_start, Stc_end = 11, 16

sst_grid = ds_nino.MLHB.loc[:, sst_start:sst_end, :, -40:40, :].mean(axis=1).rolling(lat=4, lon=4, center=True, min_periods=1).mean()
# Nsst_grid = ds_nino.MLHB.loc[:, Ntc_start:Ntc_end, :, -0:40, :].mean(axis=1)
# Ssst_grid = ds_nino.MLHB.loc[:, Stc_start:Stc_end, :, -40:0, :].mean(axis=1)
# sst_grid = xr.concat([Ssst_grid, Nsst_grid], dim='lat').rolling(lat=4, lon=4, center=True, min_periods=1).mean()

# sst_grid.loc[:, :, 0, :] = np.nan

# titles = ['D(0)JF SST of MIX', 'CP', 'EP', 'SUC']
titles = ['dTdt of MIX', 'CP', 'EP', '']
texts = alh[:3]
lon = sst_grid.lon.values
lat = sst_grid.lat.values

levels = np.linspace(-0.2, 0.2, 21)

nax = 0
cb = plot_sst(axs[nax, :], sst_grid[:, 1:4].sum(dim='term'), levels, draw_title=True, draw_text=True)
# cbaxes = fig.add_axes([0.2, 0.6, 0.6, 0.02])
cax = inset_axes(axs[nax, 2], width="4%", height="100%",  loc='lower left', bbox_to_anchor=(1.1, 0.0, 1, 1), bbox_transform=axs[nax, 2].transAxes, borderpad=0) 
cbar = plt.colorbar(cb, orientation="vertical", drawedges=True, spacing='uniform', extendrect=True, format='%4.1f', cax=cax)
# cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.tick_params(length=0)
cbar.ax.set_yticks([-0.2, -0.1, 0, 0.1, 0.2])
cbar.ax.set_ylabel('deg. C')
for spine in cbar.ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2) 
    spine.set_edgecolor('black')



nax = 1
titles = ['ADV U of MIX', 'CP', 'EP', '']
texts = alh[3:6]
cb = plot_sst(axs[nax, :], sst_grid[:, 1:2].sum(dim='term'), levels, draw_title=True, draw_text=True)
cax = inset_axes(axs[nax, 2], width="4%", height="100%",  loc='lower left', bbox_to_anchor=(1.1, 0.0, 1, 1), bbox_transform=axs[nax, 2].transAxes, borderpad=0) 
cbar = plt.colorbar(cb, orientation="vertical", drawedges=True, spacing='uniform', extendrect=True, format='%4.1f', cax=cax)
cbar.ax.tick_params(length=0)
cbar.ax.set_yticks([-0.2, -0.1, 0, 0.1, 0.2])
cbar.ax.set_ylabel('TCs')
for spine in cbar.ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2) 
    spine.set_edgecolor('black')

nax = 2
titles = ['ADV u of MIX', 'CP', 'EP', '']
cb = plot_sst(axs[nax, :], sst_grid[:, 2:3].sum(dim='term'), levels, draw_title=True, draw_text=True)
# cbaxes = fig.add_axes([0.2, 0.6, 0.6, 0.02])
cax = inset_axes(axs[nax, 2], width="4%", height="100%",  loc='lower left', bbox_to_anchor=(1.1, 0.0, 1, 1), bbox_transform=axs[nax, 2].transAxes, borderpad=0) 
cbar = plt.colorbar(cb, orientation="vertical", drawedges=True, spacing='uniform', extendrect=True, format='%4.1f', cax=cax)
# cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.tick_params(length=0)
cbar.ax.set_yticks([-0.2, -0.1, 0, 0.1, 0.2])
cbar.ax.set_ylabel('deg. C')
for spine in cbar.ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2) 
    spine.set_edgecolor('black')



nax = 3
titles = ['ADV nonlinear of MIX', 'CP', 'EP', '']
texts = alh[3:6]
cb = plot_sst(axs[nax, :], sst_grid[:, 3:4].sum(dim='term'), levels, draw_title=True, draw_text=True)
cax = inset_axes(axs[nax, 2], width="4%", height="100%",  loc='lower left', bbox_to_anchor=(1.1, 0.0, 1, 1), bbox_transform=axs[nax, 2].transAxes, borderpad=0) 
cbar = plt.colorbar(cb, orientation="vertical", drawedges=True, spacing='uniform', extendrect=True, format='%4.1f', cax=cax)
cbar.ax.tick_params(length=0)
cbar.ax.set_yticks([-0.2, -0.1, 0, 0.1, 0.2])
cbar.ax.set_ylabel('TCs')
for spine in cbar.ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2) 
    spine.set_edgecolor('black')

# # for ax in axs[1:, :].flatten():
# #     ax.plot([0, 360], [0, 0], transform=ccrs.PlateCarree(), color='white', linestyle='-', linewidth=2)

fig.subplots_adjust(left=0.07, right=0.93, bottom=0.05, top=0.95, wspace=0.4, hspace=0.5) 
# fig.savefig('./pic-20c/composite sst TCT lat lon.png', dpi=300)
# fig.savefig('./pic-d4pdf/composite sst TCT lat lon.png', dpi=300)

In [ ]:
def plot_sst(axs, data, levels, draw_title=False, draw_text=False):
    for i, ax in enumerate(axs.flatten()):
        ax.set_aspect('auto')
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1.2) 
            spine.set_edgecolor('black')
        if draw_title:
            ax.set_title(titles[i], fontsize=12)
        if draw_text:
            ax.text(-0.02, 1.05, texts[i], transform=ax.transAxes, fontsize=16, fontweight='bold')
        try:
            cb = ax.contourf(lon, mon[5:], data[i, 5:], levels=levels, cmap=cmaps.hotcold_18lev, extend='both')
        except:
            cb = ax.contourf(lon[:-1], mon[5:], data[i, 5:], levels=levels, cmap=cmaps.hotcold_18lev, extend='both')
        ytick = np.array([-1, 5, 11, 17, 23])
        ax.set_yticks(ytick)
        ax.set_yticklabels(months_str[ytick+6], fontsize=8) 
        ax.set_xticks(np.arange(120, 271, 30))
        ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
        ax.tick_params(axis='both', which='major', direction='out', length=5, width=1.2, labelsize=9, colors='black')
    return cb


ds_nino = xr.open_dataset('./data/composite_dTdt.nc')
fig, axs = plt.subplots(4, 3, figsize=(9, 6))
mon = ds_nino.month.values
sum_mon = np.r_[mon[0:5], mon[11:17], mon[23:29]]
win_mon = np.r_[mon[5:11], mon[17:23], mon[29: ]]
sst_grid = ds_nino.MLHB.loc[:, :, :, -5:5, :].mean(dim='lat').rolling(lon=4, month=3, center=True, min_periods=1).mean()


months_str = np.array(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
months_str = np.array([f'{months_str[mon[i]]} -1' for i in range(6)] + [f'{months_str[mon[i]]} 0' for i in range(6, 18)] + [f'{months_str[mon[i]-12]} 1' for i in range(18, 30)] + [f'{months_str[mon[i]-24]} 2' for i in range(30, 36)])
lon = sst_grid.lon.values

levels = np.linspace(-0.4, 0.4, 21)

titles = ['ADV w of MIX', 'CP', 'EP', 'SUC']
texts = alh[:3]
cb = plot_sst(axs[0, :], sst_grid[: ,:, 7:10].sum(dim='term'), levels, draw_title=True, draw_text=True)
cax = inset_axes(axs[0, 2], width="4%", height="100%",  loc='lower left', bbox_to_anchor=(1.1, 0, 1, 1), bbox_transform=axs[0, 2].transAxes, borderpad=0) 
cbar = plt.colorbar(cb, orientation="vertical", drawedges=True, spacing='uniform', extendrect=True, format='%4.1f', cax=cax)
# cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.tick_params(length=0)
cbar.ax.set_yticks([-0.2, -0.1, 0, 0.1, 0.2])
cbar.ax.set_ylabel('deg. C')
for spine in cbar.ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2) 
    spine.set_edgecolor('black')

nax = 1
titles = ['-wTz of MIX', 'CP', 'EP', 'SUC']
texts = alh[3:6]
cb = plot_sst(axs[nax, :], sst_grid[: ,:, 7:8].sum(dim='term'), levels, draw_title=True, draw_text=True)
cax = inset_axes(axs[nax, 2], width="4%", height="100%",  loc='lower left', bbox_to_anchor=(1.1, 0, 1, 1), bbox_transform=axs[nax, 2].transAxes, borderpad=0) 
cbar = plt.colorbar(cb, orientation="vertical", drawedges=True, spacing='uniform', extendrect=True, format='%4.1f', cax=cax)
cbar.ax.tick_params(length=0)
cbar.ax.set_yticks([-0.2, -0.1, 0, 0.1, 0.2])
cbar.ax.set_ylabel('deg. C')
for spine in cbar.ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2) 
    spine.set_edgecolor('black')
# for ax in axs[nax, :]:
#     ax.add_patch(Ellipse(xy=(0.6, 0.5), width=0.7, height=0.2, angle=55, linewidth=1.5,
#                      edgecolor='red', facecolor='none', alpha=0.8, transform=ax.transAxes))

nax = 2
titles = ['-Wtz of MIX', 'CP', 'EP', 'SUC']
texts = alh[6:9]
cb = plot_sst(axs[nax, :], sst_grid[: ,:, 8:9].sum(dim='term'), levels, draw_title=True, draw_text=True)
# cb = plot_sst(axs[1, :], tcg_grid, level_tcg)
cax = inset_axes(axs[nax, 2], width="4%", height="100%",  loc='lower left', bbox_to_anchor=(1.1, 0, 1, 1), bbox_transform=axs[nax, 2].transAxes, borderpad=0) 
cbar = plt.colorbar(cb, orientation="vertical", drawedges=True, spacing='uniform', extendrect=True, format='%4.1f', cax=cax)
cbar.ax.tick_params(length=0)
cbar.ax.set_yticks([-0.2, -0.1, 0, 0.1, 0.2])
cbar.ax.set_ylabel('deg. C')
for spine in cbar.ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2) 
    spine.set_edgecolor('black')


nax = 3
titles = ['-wtz of MIX', 'CP', 'EP', 'SUC']
texts = alh[9:12]
cb = plot_sst(axs[nax, :], sst_grid[: ,:, 9:10].sum(dim='term'), levels, draw_title=True, draw_text=True)
# cb = plot_sst(axs[1, :], tcg_grid, level_tcg)
cax = inset_axes(axs[nax, 2], width="4%", height="100%",  loc='lower left', bbox_to_anchor=(1.1, 0, 1, 1), bbox_transform=axs[nax, 2].transAxes, borderpad=0) 
cbar = plt.colorbar(cb, orientation="vertical", drawedges=True, spacing='uniform', extendrect=True, format='%4.1f', cax=cax)
cbar.ax.tick_params(length=0)
cbar.ax.set_yticks([-0.2, -0.1, 0, 0.1, 0.2])
cbar.ax.set_ylabel('deg. C')
for spine in cbar.ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2) 
    spine.set_edgecolor('black')

fig.subplots_adjust(left=0.07, right=0.9, bottom=0.05, top=0.95, wspace=0.4, hspace=0.5) 
fig.savefig('./pic-mlhb/dt w.png')


In [ ]:
def plot_sst(axs, data, levels, draw_title=False, draw_text=False):
    for i, ax in enumerate(axs.flatten()):
        ax.set_aspect('auto')
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1.2) 
            spine.set_edgecolor('black')
        if draw_title:
            ax.set_title(titles[i], fontsize=12)
        if draw_text:
            ax.text(-0.02, 1.05, texts[i], transform=ax.transAxes, fontsize=16, fontweight='bold')
        try:
            cb = ax.contourf(lon, mon[5:], data[i, 5:], levels=levels, cmap=cmaps.hotcold_18lev, extend='both')
        except:
            cb = ax.contourf(lon[:-1], mon[5:], data[i, 5:], levels=levels, cmap=cmaps.hotcold_18lev, extend='both')
        ytick = np.array([-1, 5, 11, 17, 23])
        ax.set_yticks(ytick)
        ax.set_yticklabels(months_str[ytick+6], fontsize=8) 
        ax.set_xticks(np.arange(120, 271, 30))
        ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
        ax.tick_params(axis='both', which='major', direction='out', length=5, width=1.2, labelsize=9, colors='black')
    return cb


ds_nino = xr.open_dataset('./data/composite_dTdt.nc')
fig, axs = plt.subplots(4, 3, figsize=(9, 6))
mon = ds_nino.month.values
sum_mon = np.r_[mon[0:5], mon[11:17], mon[23:29]]
win_mon = np.r_[mon[5:11], mon[17:23], mon[29: ]]
sst_grid = ds_nino.MLHB.loc[:, :, :, -5:5, :].mean(dim='lat').rolling(lon=4, month=3, center=True, min_periods=1).mean()

dire = 'w'
u = ds_nino.u.loc[:, :, dire, -5:5, :].mean(dim='lat')
U = ds_nino.U.loc[:, :, dire, -5:5, :].mean(dim='lat')
tx = ds_nino.dt.loc[:, :, dire, -5:5, :].mean(dim='lat').rolling(lon=4, month=3, center=True, min_periods=1).mean()
Tx = ds_nino.dT.loc[:, :, dire, -5:5, :].mean(dim='lat')

months_str = np.array(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
months_str = np.array([f'{months_str[mon[i]]} -1' for i in range(6)] + [f'{months_str[mon[i]]} 0' for i in range(6, 18)] + [f'{months_str[mon[i]-12]} 1' for i in range(18, 30)] + [f'{months_str[mon[i]-24]} 2' for i in range(30, 36)])
lon = sst_grid.lon.values


# levels = np.linspace(-0.5, 0.5, 21)
levels = np.linspace(-0.5, 0.5, 21)
titles = ['w of MIX', 'CP', 'EP', 'SUC']
texts = alh[:3]
cb = plot_sst(axs[0, :], u*1e+5, levels, draw_title=True, draw_text=True)
cax = inset_axes(axs[0, 2], width="4%", height="100%",  loc='lower left', bbox_to_anchor=(1.1, 0, 1, 1), bbox_transform=axs[0, 2].transAxes, borderpad=0) 
cbar = plt.colorbar(cb, orientation="vertical", drawedges=True, spacing='uniform', extendrect=True, format='%4.1f', cax=cax)
# cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.tick_params(length=0)
# cbar.ax.set_yticks([-0.2, -0.1, 0, 0.1, 0.2])
cbar.ax.set_ylabel('deg. C')
for spine in cbar.ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2) 
    spine.set_edgecolor('black')

nax = 1
titles = ['W of MIX', 'CP', 'EP', 'SUC']
texts = alh[3:6]
cb = plot_sst(axs[nax, :], U*1e+5, levels, draw_title=True, draw_text=True)
cax = inset_axes(axs[nax, 2], width="4%", height="100%",  loc='lower left', bbox_to_anchor=(1.1, 0, 1, 1), bbox_transform=axs[nax, 2].transAxes, borderpad=0) 
cbar = plt.colorbar(cb, orientation="vertical", drawedges=True, spacing='uniform', extendrect=True, format='%4.1f', cax=cax)
cbar.ax.tick_params(length=0)
# cbar.ax.set_yticks([-0.2, -0.1, 0, 0.1, 0.2])
cbar.ax.set_ylabel('deg. C')
for spine in cbar.ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2) 
    spine.set_edgecolor('black')
# for ax in axs[nax, :]:
#     ax.add_patch(Ellipse(xy=(0.6, 0.5), width=0.7, height=0.2, angle=55, linewidth=1.5,
#                      edgecolor='red', facecolor='none', alpha=0.8, transform=ax.transAxes))

nax = 2
levels = np.linspace(-1, 1, 21)
titles = ['dtdz of MIX', 'CP', 'EP', 'SUC']
texts = alh[6:9]
cb = plot_sst(axs[nax, :], tx*15, levels, draw_title=True, draw_text=True)
# cb = plot_sst(axs[1, :], tcg_grid, level_tcg)
cax = inset_axes(axs[nax, 2], width="4%", height="100%",  loc='lower left', bbox_to_anchor=(1.1, 0, 1, 1), bbox_transform=axs[nax, 2].transAxes, borderpad=0) 
cbar = plt.colorbar(cb, orientation="vertical", drawedges=True, spacing='uniform', extendrect=True, format='%4.1f', cax=cax)
cbar.ax.tick_params(length=0)
# cbar.ax.set_yticks([-0.2, -0.1, 0, 0.1, 0.2])
cbar.ax.set_ylabel('deg. C')
for spine in cbar.ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2) 
    spine.set_edgecolor('black')


nax = 3
titles = ['dTdz of MIX', 'CP', 'EP', 'SUC']
texts = alh[9:12]
cb = plot_sst(axs[nax, :], Tx*15, levels, draw_title=True, draw_text=True)
# cb = plot_sst(axs[1, :], tcg_grid, level_tcg)
cax = inset_axes(axs[nax, 2], width="4%", height="100%",  loc='lower left', bbox_to_anchor=(1.1, 0, 1, 1), bbox_transform=axs[nax, 2].transAxes, borderpad=0) 
cbar = plt.colorbar(cb, orientation="vertical", drawedges=True, spacing='uniform', extendrect=True, format='%4.1f', cax=cax)
cbar.ax.tick_params(length=0)
# cbar.ax.set_yticks([-0.2, -0.1, 0, 0.1, 0.2])
cbar.ax.set_ylabel('deg. C')
for spine in cbar.ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2) 
    spine.set_edgecolor('black')

fig.subplots_adjust(left=0.07, right=0.93, bottom=0.05, top=0.95, wspace=0.4, hspace=0.5) 
fig.savefig('./pic-mlhb/z.png')


In [ ]:
def plot_sst(axs, data, levels, draw_title=False, draw_text=False):
    for i, ax in enumerate(axs.flatten()):
        ax.set_aspect('auto')
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1.2) 
            spine.set_edgecolor('black')
        if draw_title:
            ax.set_title(titles[i], fontsize=11)
        if draw_text:
            ax.text(-0.07, 1.08, texts[i], transform=ax.transAxes, fontsize=16, fontweight='bold')
        try:
            cb = ax.contourf(lon, mon[5:], data[i, 5:], levels=levels, cmap=cmaps.hotcold_18lev, extend='both')
        except:
            cb = ax.contourf(lon[:-1], mon[5:], data[i, 5:], levels=levels, cmap=cmaps.hotcold_18lev, extend='both')
        ytick = np.array([-1, 5, 11, 17, 23, 29])
        ax.set_yticks(ytick)
        ax.set_yticklabels(months_str[ytick+6], fontsize=8) 
        ax.set_xticks(np.arange(120, 271, 30))
        ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
        ax.tick_params(axis='both', which='major', direction='out', length=5, width=1.2, labelsize=9, colors='black')
    return cb

def plot_box1(ax):
    x1, x2 = 230, 275
    y1, y2 = 17, 25
    ax.plot([x1, x2, x2, x1, x1], [y1, y1, y2, y2, y1], lw=2.5, ls='--', color='#713E50', zorder=2)
def plot_box2(ax):
    x1, x2 = 150, 205
    y1, y2 = 5, 15
    ax.plot([x1, x2, x2, x1, x1], [y1, y1, y2, y2, y1], lw=2.5, ls='--', color='#E9D5DB', zorder=2)
def plot_box3(ax):
    x1, x2 = 210, 270
    y1, y2 = 3, 15
    ax.plot([x1, x2, x2, x1, x1], [y1, y1, y2, y2, y1], lw=2.5, ls='--', color='#526879', zorder=2)



ds_nino = xr.open_dataset('./data/composite_dTdt.nc')
fig, axs = plt.subplots(4, 2, figsize=(7, 6))
mon = ds_nino.month.values
sum_mon = np.r_[mon[0:5], mon[11:17], mon[23:29]]
win_mon = np.r_[mon[5:11], mon[17:23], mon[29: ]]
sst_grid = ds_nino.MLHB.loc['MIX', :, :, -5:5, :].mean(dim='lat').rolling(lon=4, month=3, center=True, min_periods=1).mean()

dire = 'u'
u = ds_nino.u.loc['MIX', :, dire, -5:5, :].mean(dim='lat')
U = ds_nino.U.loc['MIX', :, dire, -5:5, :].mean(dim='lat')
tx = ds_nino.dt.loc['MIX', :, dire, -5:5, :].mean(dim='lat').rolling(lon=4, month=3, center=True, min_periods=1).mean()
Tx = ds_nino.dT.loc['MIX', :, dire, -5:5, :].mean(dim='lat')

months_str = np.array(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
months_str = np.array([f'{months_str[mon[i]]} -1' for i in range(6)] + [f'{months_str[mon[i]]} 0' for i in range(6, 18)] + [f'{months_str[mon[i]-12]} 1' for i in range(18, 30)] + [f'{months_str[mon[i]-24]} 2' for i in range(30, 36)])
lon = sst_grid.lon.values

levels = np.linspace(-0.5, 0.5, 21)
titles = ['Zonal advection term of MLHB', r"$-U \partial t/\partial x$", r"$-u \partial T/\partial x$", r"$-u \partial t/\partial x$"]
texts = 'abcd'
cb = plot_sst(axs[:, 0], np.transpose(np.concatenate([sst_grid[:, 1:4].sum(dim='term', keepdims=True), sst_grid[:, 1:4]], axis=1), (1, 0, 2)), levels, draw_title=True, draw_text=True)

titles = [r"$u$", r"$U$", r"$\partial t/\partial x$", r"$\partial T/\partial x$"]
texts = 'efgh'
data = np.array([u.values, U.values, tx.values*5e+5, Tx.values*5e+5])
cb = plot_sst(axs[:, 1], data, levels, draw_title=True, draw_text=True)

cbaxes = fig.add_axes([0.2, 0.04, 0.6, 0.01])
cb = plt.colorbar(cb, cax=cbaxes, orientation="horizontal")

plot_box1(axs[0, 0])
plot_box2(axs[0, 0])
plot_box3(axs[0, 0])

plot_box1(axs[1, 0])
plot_box2(axs[1, 0])

plot_box3(axs[2, 0])

plot_box1(axs[3, 0])
plot_box2(axs[3, 0])

plot_box1(axs[0, 1])
plot_box2(axs[0, 1])
plot_box3(axs[0, 1])

plot_box1(axs[1, 1])
plot_box2(axs[1, 1])

plot_box1(axs[2, 1])
plot_box2(axs[2, 1])

plot_box3(axs[3, 1])
fig.subplots_adjust(left=0.07, right=0.93, bottom=0.1, top=0.95, wspace=0.4, hspace=0.5)

fig.savefig('./pic-mlhb/adv u.png', dpi=300)

In [ ]:
ds_nino

In [ ]:
sst_2026 = xr.DataArray(np.empty((12, 180, 360)), coords=[np.arange(1, 13), np.arange(89.5, -90, -1), np.arange(-179.5, 180, 1)], dims=['month', 'lat', 'lon'])
f = open(r'/Users/jiangdi/Downloads/HadISST1_SST_2025.txt')
data_str = f.readlines()
n = -1

for i in range(len(data_str)):
    data = data_str[i][:-1].replace('-', ' -').split()
    if data[-1] == 'columns':
        n += 1
        rows = -1
        print(data)
    else:
        if len(data) == 360:
            rows += 1
            sst_2026[n, rows, :] = np.array(data, dtype=np.int64)



In [ ]:
np.arange(89.5, -90, 1).shape

In [ ]:
sst_2026_ = sst_2026.where(np.logical_and(abs(sst_2026) > 1000, abs(sst_2026) < 10000))/100
sst_2026_.loc[:, 5:-5, :].mean(dim='lat').plot()

In [ ]:
tsurfa = xr.open_dataset(r'/Volumes/shiki/study/data/SST/202609/tsurfa.9.adj.1sep2026.24m.nc').tsurfa

In [ ]:
tsurfa.loc[:, :, -5:5, :].mean(dim='lat').plot.contourf(col='lev', col_wrap=2, size=2, aspect=2, levels=21)

In [ ]:
tsurfa[:, 0, -40:40, :]